# 1. Root Finding Methods

Finding roots of $f(x) = 0$ is fundamental in numerical analysis. This notebook covers:
- **Bisection method**: guaranteed convergence, slow
- **Newton-Raphson method**: fast (quadratic) convergence, needs derivative
- **Secant method**: no derivative needed, superlinear convergence
- Convergence comparison and error plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Test function: f(x) = x^3 - 2x - 5, root near x ~ 2.0946
def f(x):
    return x**3 - 2*x - 5

def f_prime(x):
    return 3*x**2 - 2

x = np.linspace(1, 3, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, f(x), 'b-', lw=2)
ax.axhline(0, color='k', ls='--', lw=0.8)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('$f(x) = x^3 - 2x - 5$')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.2 Bisection Method

Given $f(a) \cdot f(b) < 0$, the bisection method repeatedly halves the interval $[a, b]$.

**Convergence**: linear, error bounded by $(b-a)/2^n$ after $n$ iterations.

In [ ]:
def bisection(f, a, b, tol=1e-12, max_iter=100):
    errors = []
    for i in range(max_iter):
        c = (a + b) / 2
        errors.append(abs(b - a) / 2)
        if abs(f(c)) < tol or (b - a) / 2 < tol:
            return c, errors
        if f(a) * f(c) < 0:
            b = c
        else:
            a = c
    return c, errors

root_bis, errors_bis = bisection(f, 2.0, 3.0)
print(f"Bisection root: {root_bis:.12f}")
print(f"f(root) = {f(root_bis):.2e}")
print(f"Iterations: {len(errors_bis)}")

## 1.3 Newton-Raphson Method

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

**Convergence**: quadratic near simple roots (error roughly squares each step).

In [ ]:
def newton(f, f_prime, x0, tol=1e-12, max_iter=100):
    errors = []
    x = x0
    for i in range(max_iter):
        fx = f(x)
        fpx = f_prime(x)
        if abs(fpx) < 1e-15:
            break
        x_new = x - fx / fpx
        errors.append(abs(x_new - x))
        if abs(x_new - x) < tol:
            return x_new, errors
        x = x_new
    return x, errors

root_new, errors_new = newton(f, f_prime, 2.5)
print(f"Newton root: {root_new:.12f}")
print(f"f(root) = {f(root_new):.2e}")
print(f"Iterations: {len(errors_new)}")

## 1.4 Secant Method

Approximates the derivative using two previous points:
$$x_{n+1} = x_n - f(x_n) \frac{x_n - x_{n-1}}{f(x_n) - f(x_{n-1})}$$

**Convergence**: superlinear (order $\approx 1.618$, the golden ratio).

In [ ]:
def secant(f, x0, x1, tol=1e-12, max_iter=100):
    errors = []
    for i in range(max_iter):
        f0, f1 = f(x0), f(x1)
        if abs(f1 - f0) < 1e-15:
            break
        x2 = x1 - f1 * (x1 - x0) / (f1 - f0)
        errors.append(abs(x2 - x1))
        if abs(x2 - x1) < tol:
            return x2, errors
        x0, x1 = x1, x2
    return x1, errors

root_sec, errors_sec = secant(f, 2.0, 3.0)
print(f"Secant root: {root_sec:.12f}")
print(f"f(root) = {f(root_sec):.2e}")
print(f"Iterations: {len(errors_sec)}")

## 1.5 Convergence Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(range(len(errors_bis)), errors_bis, 'o-', label='Bisection', markersize=3)
ax.semilogy(range(len(errors_new)), errors_new, 's-', label='Newton-Raphson', markersize=4)
ax.semilogy(range(len(errors_sec)), errors_sec, '^-', label='Secant', markersize=4)
ax.set_xlabel('Iteration')
ax.set_ylabel('Error (log scale)')
ax.set_title('Convergence Comparison of Root-Finding Methods')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

| Method | Convergence | Derivative needed? | Guaranteed? |
|--------|------------|-------------------|-------------|
| Bisection | Linear ($O(1/2^n)$) | No | Yes (if sign change) |
| Newton-Raphson | Quadratic | Yes | No (may diverge) |
| Secant | Superlinear ($\approx 1.618$) | No | No |

- **Bisection** is robust but slow; use it for a safe initial bracket
- **Newton** is fast but requires $f'$ and a good starting point
- **Secant** balances speed and simplicity (no derivative needed)